# Fermions on the square lattice: perfect nesting, Linhard bubble, and random phase approximation (RPA)

In this tutorial we will use TRIQS and the [**Two-Particle Response-Function toolbox (TPRF)**](https://triqs.github.io/tprf/latest/) to compute;

1. the **non-interacting Green's function** of fermions on the square lattice with nearest-neighbour hopping 
   and study the Fermi surface,

2. the **non-interacting two-particle response**, also called the susceptibility bubble,

3. the **Random-Phase Approximation (RPA)** susceptibility for weak interactions, 
   studying the anti-ferromagnetic divergence at ($\pi,\pi)$.

In [ ]:
from triqs.lattice.tight_binding import TBLattice
from triqs_tprf.lattice_utils import k_space_path
from triqs.gf import MeshImFreq

from triqs_tprf.lattice import lattice_dyson_g0_wk
from triqs_tprf.lattice_utils import k_space_path, imtime_bubble_chi0_wk
from triqs_tprf.lattice import solve_rpa_PH

from h5 import HDFArchive

In [ ]:
import numpy as np

In [ ]:
%matplotlib inline
%config InlineBackend.figure_format = 'svg'
from triqs.plot.mpl_interface import plt

## Square lattice with nearest-neighbour hopping

The square lattice with nearest-neighbour hopping $t$ appeared in earier tutorials, were the dispersion relation 

\begin{equation}
  \epsilon(\mathbf{k})=-2t(\cos{k_x}+\cos{k_y}),
\end{equation}

was computed using TRIQS in more than one way.

However, in TRIQS and TPRF there are a number of helper routines for lattice models that simplifies the study of general tight-binding models. Here we will therefore use these standard routines.

Insead of constructing $\epsilon(\mathbf{k})$ directly in momentum space we construct a real-space tight binding lattice Hamitonian $H(\mathbf{r})$ using [the `TBLattice` class](https://triqs.github.io/triqs/latest/documentation/python_api/triqs.lattice.tight_binding.TBLattice.html?highlight=tblattice#triqs.lattice.tight_binding.TBLattice) corresponding to the square lattice with nearest neighbour hopping $t=1$.

In [ ]:
T    = 0.4      # temperature
beta = 1. / T   # inverse temperature
t    = 1.0      # hopping amplitude

In [ ]:
H_r = TBLattice( 
    units=[
        (1,0,0), # basis vector in the x-direction 
        (0,1,0), # basis vector in the y-direction
    ],
    hoppings={
        (+1,0) : [[-t]], # hopping in the +x direction
        (-1,0) : [[-t]], # hopping in the -x direction
        (0,+1) : [[-t]], # hopping in the +y direction
        (0,-1) : [[-t]], # hopping in the -y direction
    })

The real-space Hamiltonian $H(\mathbf{r})$ can both construct a discretized momentum mesh and evaluate the dispersion $\epsilon(\mathbf{k})$ on a given momentum mesh, by

In [ ]:
n_k = 128
kmesh = H_r.get_kmesh(n_k=n_k)
e_k = H_r.fourier(kmesh)

Since the square lattice is two-dimensional it is possible to visualize the dispersion using a color plot. Here is an example that plots $\epsilon(\mathbf{k})$ in the entire Brillouin zone as well as the shape of the Fermi surface at $\omega = 0$ (dotted line):

In [ ]:
k = np.linspace(-np.pi, np.pi, num=100)
kx, ky = np.meshgrid(k, k)

e_k_interp = np.vectorize(lambda kx, ky : e_k((kx, ky, 0)).real)(kx, ky)

plt.figure()

plt.pcolormesh(kx, ky, e_k_interp, rasterized=True, cmap='RdBu')
plt.colorbar().ax.set_ylabel(r'$\epsilon(\mathbf{k})$'); 

plt.contour(kx, ky, e_k_interp, levels=[0], linestyles='dotted')

plt.xlabel(r'$k_x$'); plt.ylabel(r'$k_y$');
k_ticks, k_labels = [-np.pi, 0, np.pi], [r"$-\pi$", r"0", r"$\pi$"]
plt.xticks(k_ticks, k_labels); plt.yticks(k_ticks, k_labels);
plt.axis('square');

# -- High-symmetry path G-X-M-G

pts = [
    (0., 0., r'$\Gamma$', 'w', 'bottom', 'right'), 
    (np.pi, 0., r'$X$', 'k', 'center', 'left'),
    (np.pi, np.pi, r'$M$', 'r', 'bottom', 'left'),
    ]
for x, y, label, color, va, ha in pts:
    plt.plot(x, y, 'o', color=color, clip_on=False, zorder=110)
    plt.text(x, y, label, color=color, fontsize=18, va=va, ha=ha)

X, Y, _, _, _, _ = zip(*(pts+[pts[0]]))
plt.plot(X, Y, '-m', zorder=100, lw=4);

Momentum dependent quantities can also be visualized along high-symmetry paths in the Brillouin zone, see above for the high-symmetry points $\Gamma$, $X$ and $M$ of the square lattice. 

Here is an example that plots the dispersion $\epsilon(\mathbf{k})$ along thepath $\Gamma - X - M - \Gamma$ in k-space using [the `triqs_tprf.lattice_utils.k_space_path` function](https://triqs.github.io/tprf/unstable/reference/python_reference.html#triqs_tprf.lattice_utils.k_space_path).

In [ ]:
G = [0.0, 0.0, 0.0]
X = [0.5, 0.0, 0.0]
M = [0.5, 0.5, 0.0]

path = [(G, X), (X, M), (M, G)]

k_vecs, k_plot, k_ticks = k_space_path(path, num=32, bz=H_r.bz)

e_k_interp = np.vectorize(lambda k : e_k(k).real, signature='(n)->()')

plt.plot(k_plot, e_k_interp(k_vecs))
plt.xticks(k_ticks, labels=[r'$\Gamma$', '$X$', '$M$', r'$\Gamma$'])
plt.ylabel(r'$\epsilon(\mathbf{k})$')
plt.grid(True)

In the following we will re-purpose these visualization scripts to study the one-particle and two-particle Green's functions of the square lattice model.

## Non-interacting lattice Green's function

Given the dispersion $\epsilon(\mathbf{k})$ the non-interacting Green's function $G_0(i\omega_n, \mathbf{k})$ is given by

\begin{equation}
  G_0(i\omega_n, \mathbf{k}) = \frac{1}{i\omega_n - \epsilon(\mathbf{k})}
  \, .
\end{equation}

As shown in the Basic Tutorial it is of course possible to compute $G_0$ using a loop over frequency and momentum:

```python
from triqs.gf import Gf, MeshImFreq, MeshProduct

wmesh = MeshImFreq(beta=2.5, statistic='Fermion', n_max=128)
wkmesh = MeshProduct(wmesh, kmesh)
g0_wk = Gf(mesh=wkmesh, target_shape=e_k.target_shape)

for w, k in wkmesh:    
    g0_wk[w, k] = 1/(w - e_k[k])
```

However, TPRF has Dyson equation solvers that are OpenMP+MPI parallell and all implemented in C++, see [the TPRF documentation](https://triqs.github.io/tprf/latest/reference/python_reference.html#lattice-green-s-functions). Here we will use these fast routines!

Here, we use [`triqs_tprf.lattice.lattice_dyson_g0_wk`](https://triqs.github.io/tprf/latest/reference/python_reference.html#triqs_tprf.lattice.lattice_dyson_g0_wk) to compute $G_0(i\omega_n, \mathbf{k})$ at inverse temperature $\beta = 2.5$ using a [fermionic `MeshImFreq` frequency mesh](https://triqs.github.io/triqs/latest/documentation/python_api/triqs.gf.meshes.MeshImFreq.html?highlight=meshimfreq#triqs.gf.meshes.MeshImFreq)  with 128 Matsubara frequencies and name the resulting Green's function `g0_wk`:

In [ ]:
wmesh = MeshImFreq(beta=beta, statistic='Fermion', n_iw=20)
g0_wk = lattice_dyson_g0_wk(mu=0., e_k=e_k, mesh=wmesh)

## Fermi surface nesting

We will now study the Fermi surface of Fermions on the square lattice with nearest neighbour hopping, which has a special property called *perfect nesting*. A Fermi surface is said to be *nested* if parts of the Fermi surface map to each other by a single momentum vector $\mathbf{Q}$, called the *nesting vector*.

For a non-interacting system the Fermi surface is the surface in k-space defined by

$$ \epsilon(\mathbf{k}) - \mu = 0 \, ,$$

where $\mu$ is the chemical potential. In terms of the spectral function $A(\omega, \mathbf{k})$ this corresponds to large values of $A$ at $\omega=0$

$$ A(\omega = 0, \mathbf{k}) = -\frac{1}{\pi} \text{Im} 
\left[ \frac{1}{ 0 - \epsilon(\mathbf{k}) + \mu - i\delta } \right] \gg 1 \, ,$$

which also generalizes to interacting systems.

We now make a color plot of the zero-frequency spectral function $A(k, \omega=0)$ over the Brillouin zone, using the approximation

$$ A(k, \omega=0) \approx -\frac{1}{\pi} \text{Im}[ G_0(\mathbf{k}, i\omega_0) ] \, ,$$

where we neglect the fact that the first fermionic Matsubara frequency $i\omega_0$ is not exactly $0$.

The right hand side can be evaluated using the Triqs Green's function `g0_wk` and the interpolation feature:

```python
n = 0 # Matsubara frequency index
kx, ky, kz = 0., 0., 0.
k_vec = (kx, ky, kz)
g0_wk(n, k_vec)
```

In [ ]:
kgrid1d = np.linspace(-np.pi, np.pi, n_k + 1, endpoint=True)
kx, ky = np.meshgrid(kgrid1d, kgrid1d)

A_k = np.vectorize(lambda kx, ky: -g0_wk( 0, (kx, ky, 0) ).imag / np.pi)
A_inv_k = np.vectorize(lambda kx, ky: (1 / g0_wk( 0, (kx, ky, 0) )).real)

plt.contour(kx, ky, A_inv_k(kx, ky), levels=[0], colors='white')
plt.pcolormesh(kx, ky, A_k(kx, ky), rasterized=True)

plt.colorbar().ax.set_ylabel(r"$G_0(i\omega_0, \mathbf{k})$")
plt.xticks([-np.pi, 0, np.pi],[r"$-\pi$", r"0", r"$\pi$"])    
plt.yticks([-np.pi, 0, np.pi],[r"$-\pi$", r"0", r"$\pi$"])
plt.axis('square'); plt.xlabel(r"$k_x$"); plt.ylabel(r"$k_y$");

**Questions**

  * How can we see from the plot that the Fermi surface is nested?
  * What is the nesting vector?
  * Actually the Fermi surface is **perfectly** nested. What do you think is the difference between *nesting* and *perfect nesting*?

# Susceptibility $\chi_0(\mathbf{q},i\Omega_n)$ of non-interacting fermions

As we have seen in the lecture, the Lindhard bubble in Matsubara space $\chi_0(\mathbf{q},i\Omega_n)$ becomes a double convolution

\begin{equation}
  \chi_0(\mathbf{q}, i\Omega_n) = 
    2\frac{T}{N}\sum_{\mathbf{k}, m} 
    G_0(\mathbf{k}, i\omega_m)G_0(\mathbf{k}+\mathbf{q}, i\omega_m + i\Omega_n)
    \, ,
\end{equation}

where $\mathbf{q}$ and $\mathbf{k}$ are momenta and $i\Omega_n$ and $i\omega_m$ are bosonic and fermionic Matsubara frequencies, respectively, N the number of $\mathbf{k}$ points, and $T$ is the temperature.

## Fast calculation using Fourier transform

To compute $\chi_0$ the fastest approach is to perform the direct product in real space and imaginary time above. However, we usually have the single particle Green's function in momentum and frequency $G(\mathbf{k}, i\omega_n)$ and are interested in the susceptibiltiy $\chi_0$ in the same space $\chi_0(\mathbf{q}, i\Omega_n)$.

Therefore we compute $\chi_0$ by Fourier transforming the Green's function $G$ to imaginary time $\tau$ and real space $\mathbf{r}$, using fast Fourier transforms (FFT)

$$
G_0(\mathbf{r}, \tau) = 
  \mathcal{F}_{\{\mathbf{k}, i\nu_m\} \rightarrow \{\mathbf{r}, \tau\}} 
  \big\{ G_0(\mathbf{k}, i\omega_n) \big\}
  \, ,
$$

giving $\chi_0$ as the simple product

$$
\chi_0(\mathbf{r},\tau) = 2 G_0(\mathbf{r},\tau)G_0(-\mathbf{r},\beta -\tau)
\, ,
$$

where we have added a factor of 2 for spin. Finally we Fourier transform $\chi_0$ back to momentum and Matsubara frequency

$$ 
  \chi_0(\mathbf{q},i\omega_n) \equiv 
  \mathcal{F}_{\{\mathbf{r},\tau\} \rightarrow \{\mathbf{q}, i\omega_n\}} 
  \big\{ \chi_0(\mathbf{r}, \tau) \big\}
  \, .
$$

## Compute the susceptibility $\chi_0(\mathbf{q}, i\omega_n)$

The effective Fourier transform based routine for computing the susceptibility is implemented in TPRF in the function `triqs_trpf.lattice_utils.imtime_bubble_chi0_wk`. To construct $\chi_0$ as defined above it can be called using the lattice Green's function `g0_wk`.

```python
from triqs_tprf.lattice_utils import imtime_bubble_chi0_wk
chi0_wk = 2 * imtime_bubble_chi0_wk(g0_wk, nw=100) # Factor of 2 for spin
```

We now compute `chi0_wk` using this routine.

In [ ]:
chi0_wk = 2 * imtime_bubble_chi0_wk(g0_wk, nw=10)

## Static bubble susceptibility $\chi_0(\mathbf{q}, i\Omega_n=0)$ and perfect nesting

We already saw that the square lattice with nearest-neighbour hopping $t$ has a perfectly nested Fermi surface at half-filling. In other words, large parts of the Fermi surface are mapped on to each other by a single momentum transfer $\mathbf{Q}$, called the *nesting vector*. Go back to the previous notebook and the plot of $-\frac{1}{\pi} \text{Im } G_0(\mathbf{k}, i\omega_0)$ and remind your self about the nesting vector $\mathbf{Q} = (\pi, \pi)$. The perfect nesting greatly enhances the static susceptibility, which has a dominant peak at $\chi_0(\mathbf{Q}, i\Omega_n=0)$.

We now investigate the momentum structure of the susceptibility $\chi_0$ by making a two-dimensional color plot of $\chi_0(\mathbf{q}, i\Omega_n=0)$ over the Brillouin zone.

In [ ]:
k = np.linspace(0, 2*np.pi, num=100, endpoint=True)
kx, ky = np.meshgrid(k, k)

chi_interp = np.vectorize(lambda kx, ky: chi0_wk(0, (kx, ky, 0)).real)

plt.pcolormesh(kx, ky, chi_interp(kx, ky), rasterized=True)

plt.title('Static susceptibility $\chi_0(\mathbf{q}, i\Omega_n=0)$')
ticks, labels = [0, np.pi, 2*np.pi], [r"0",r"$\pi$",r"$2\pi$"]
plt.xticks(ticks, labels); plt.yticks(ticks, labels);
plt.xlabel(r'$q_x$'); plt.ylabel(r'$q_y$')
plt.colorbar();

**Questions**

- Does the extrema of $\chi_0$ appear at the suggested nesting vector $\mathbf{Q}$?
- Is it possible to see any additional momentum structure in $\chi_0$?

We further plot the static susceptibility $\chi_0(\mathbf{q}, i\Omega_n=0)$ along the high symmetry path
$\Gamma \rightarrow X \rightarrow M \rightarrow \Gamma$ in the Brillouin zone. 

In [ ]:
G = [0.0, 0.0, 0.0]
X = [0.5, 0.0, 0.0]
M = [0.5, 0.5, 0.0]

path = [(G, X), (X, M), (M, G)]

k_vecs, k_plot, k_ticks = k_space_path(path, num=32, bz=chi0_wk.mesh[1].bz)
    
chi0_k_interp = np.vectorize(lambda k : chi0_wk(0, k).real, signature='(n)->()')
 
plt.plot(k_plot, chi0_k_interp(k_vecs))
plt.xticks(k_ticks, labels=[r'$\Gamma$', '$X$', '$M$', r'$\Gamma$'])
plt.ylabel(r'$\chi_0(\mathbf{q}, i\Omega_n=0)$'); plt.grid()

**Questions**

- How is the perfect nesting manifest in the plot?
- Is it possible to understand also the additional fine structure in $\chi_0$?

### <i class="fa fa-gear fa-x" style="color: #186391"></i> Exercise 1:  The random phase approximation (RPA)

We now want to calculate the magnetic response of the *interacting* system in the random phase approximation.

In the lecture we have seen that the formula can be derived from screening processes and modified for the Hubbard model in the following way:

$$
  \chi^\text{RPA}(\mathbf{q}, i\Omega_n) = \frac{\chi_0(\mathbf{q}, i\Omega_n)}{1 - \frac{U}{2} \chi_0(\mathbf{q}, i\Omega_n)},
$$

with the bubble (convolution of two Green functions) just calcuated before in this notebook.

Calcuate the $\chi^\text{RPA}(\mathbf{q}, i\Omega_n)$ and plot it like the $\chi_0(\mathbf{q}, i\Omega_n)$. What is the difference in magnitude of the response at $\mathbf{Q}=(\pi, \pi)$ between $\chi_{RPA}$ and $\chi_0$?

In [ ]:
U = 2

chi_RPA_wk = chi0_wk.copy()

for iW, k in chi0_wk.mesh:
    chi_RPA_wk[iW, k] = chi0_wk[iW, k] / (1. - U / 2. * chi0_wk[iW, k])

In [ ]:
k = np.linspace(0, 2*np.pi, 100, endpoint=True)
kx, ky = np.meshgrid(k, k)

chi_k_interp = np.vectorize(lambda qx, qy: chi_RPA_wk(0,(qx, qy, 0)).real)

plt.pcolor(kx, ky, chi_k_interp(kx, ky), rasterized=True)

plt.title('Static RPA susceptibility $\chi^{RPA}(\mathbf{q}, \omega=0)$')
ticks, labels = [0, np.pi, 2*np.pi], [r"0",r"$\pi$",r"$2\pi$"]
plt.xticks(ticks, labels); plt.yticks(ticks, labels);
plt.xlabel(r'$q_x$'); plt.ylabel(r'$q_y$')
plt.colorbar();

In [ ]:
G = [0.0, 0.0, 0.0]
X = [0.5, 0.0, 0.0]
M = [0.5, 0.5, 0.0]

path = [(G, X), (X, M), (M, G)]

from triqs_tprf.lattice_utils import k_space_path

k_vecs, k_plot, k_ticks = k_space_path(path, num=32, bz=chi_RPA_wk.mesh[1].bz)
    
chi_RPA_k_interp = np.vectorize(lambda k : chi_RPA_wk(0, k).real, signature='(n)->()')
 
plt.plot(k_plot, chi_RPA_k_interp(k_vecs))
plt.xticks(k_ticks, labels=[r'$\Gamma$', '$X$', '$M$', r'$\Gamma$'])
plt.ylabel(r'$\chi^{RPA}(\mathbf{q}, i\Omega_n=0)$'); plt.grid()

### <i class="fa fa-gear fa-x" style="color: #186391"></i> Exercise 2:  Critical $U$

At some critical value of the interaction $U = U_c$ the RPA susceptibility diverges

$$\chi^\text{RPA} \rightarrow \infty \, .$$

To determine $U_c$ we can study the root of the inverse susceptibility $\chi_{RPA}^{-1}$.

For the square lattice it is sufficient to study the response at $\mathbf{Q}_{AF}= (\pi, \pi)$ since this is the momentum vector where the response diverges. Analytically this occurs when the denominator is zero $1 - \frac{U}{2} \chi_0(\mathbf{Q}_{AF}, 0) = 0$, i.e.

$$ U_c^\text{RPA} = \frac{2}{\chi_0(\mathbf{Q}_{AF}, 0)} $$

Note that $U_c^\text{RPA}$ only depends on $\chi_0$ (not $\chi^\text{RPA}$).

Plot $\left(\chi^\text{RPA}\right)^{-1} (\mathbf{Q}_{AF}, 0)$ vs $U$ to numerically determine the critical $U_c^\text{RPA}$ in RPA.


In [ ]:
k_AF = (np.pi, np.pi, 0)
U_vec = np.linspace(1, 4, 10)
chi_inv_vec = []

for U in U_vec:
    chi_inv_vec.append(1. / (np.squeeze(chi0_wk(0, k_AF).real / (1. - U / 2. * chi0_wk(0, k_AF).real))))

plt.plot(U_vec, chi_inv_vec, '.-', label=r'$\chi_{RPA}^{-1}$')
plt.plot(U_vec, 0 * U_vec, 'k', lw=0.5)

plt.xlabel(r'$U$'); plt.ylabel(r'$\chi_\mathrm{RPA}^{-1}$'); plt.legend();